## Set up

### libraries

In [31]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from openpyxl.utils import get_column_letter


### Data Extraction

In [32]:
def load_lift_drag_data(root_dir):
    data_by_config_aoa = defaultdict(lambda: {'lift': [], 'drag': [], 'turbulence_model': '', 'geometry': '', 'mesh': '', 'version': '', 'grid': ''})
    
    for dirpath, _, filenames in os.walk(root_dir):
        # Find lift and drag files
        lift_files = [f for f in filenames if 'lift_force' in f and f.endswith('.txt')]
        drag_files = [f for f in filenames if 'drag_force' in f and f.endswith('.txt')]
        
        if lift_files and drag_files:
            config = None
            
            # Extract configuration based on method
            if CONFIG_EXTRACTION_METHOD == 'case_file':
                # Look for .cas or .cas.h5 files and extract filename
                for filename in filenames:
                    if filename.endswith('.cas') or filename.endswith('.cas.h5'):
                        config = filename.replace('.cas.h5', '').replace('.cas', '')
                        if config and config[0].isdigit() and '.' in config:
                            break
                        else:
                            config = None
            else:
                # Legacy: parse from folder structure
                parts = dirpath.split(os.sep)
                for part in parts:
                    if part and part[0].isdigit() and part.count('.') >= 2:
                        config = part
                        break
            
            # Find AoA value from folder name (e.g., AoA_10)
            aoa = None
            parts = dirpath.split(os.sep)
            for part in parts:
                if part.startswith('AoA_'):
                    aoa = part
                    break
            
            if config and aoa:
                # Extract AoA number
                aoa_number = aoa.split('_')[1]
                
                # Check if config contains _AoA_ pattern (old format embedded in filename)
                # Example: "4.3.1.3_AoA_5" becomes "4.3.1.3.5"
                if '_AoA_' in config:
                    config = config.replace(f'_AoA_{aoa_number}', f'.{aoa_number}')
                # Check if this is old format with no dots (version name only)
                # Example: "version_name" becomes "version_name.5"
                elif config.count('.') == 0:
                    config = f"{config}.{aoa_number}"
                
                # Parse configuration using position and value mappings
                config_parts = config.split('.')
                positions = POSITION_MAP
                mappings = VALUE_MAPPINGS
                
                # Extract each field based on position
                geometry_num = config_parts[positions['geometry']] if positions['geometry'] is not None and len(config_parts) > positions['geometry'] else None
                mesh_num = config_parts[positions['mesh']] if positions['mesh'] is not None and len(config_parts) > positions['mesh'] else None
                turbulence_num = config_parts[positions['turbulence']] if positions['turbulence'] is not None and len(config_parts) > positions['turbulence'] else None
                version_num = config_parts[positions['version']] if positions['version'] is not None and len(config_parts) > positions['version'] else None
                grid_code = config_parts[positions['grid']] if positions['grid'] is not None and len(config_parts) > positions['grid'] else None
                
                # Map to descriptive names
                geometry = mappings.get('geometry', {}).get(geometry_num, f"Geometry_{geometry_num}") if geometry_num else "N/A"
                mesh = mappings.get('mesh', {}).get(mesh_num, f"Mesh_{mesh_num}") if mesh_num else "N/A"
                turbulence_model = mappings.get('turbulence', {}).get(turbulence_num, f"Turbulence_{turbulence_num}") if turbulence_num else "Unknown"
                version = mappings.get('version', {}).get(version_num, f"Version_{version_num}") if version_num else "N/A"
                grid = mappings.get('grid', {}).get(grid_code, f"Grid_{grid_code}") if grid_code else "N/A"
                
                # Extract angle of attack in degrees (e.g., AoA_10 → 10)
                aoa_degrees = float(aoa_number)
                aoa_radians = np.radians(aoa_degrees)
                
                # Read all lift files
                for lift_file in lift_files:
                    lift_path = os.path.join(dirpath, lift_file)
                    with open(lift_path) as f:
                        lift_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    lift_data.append(value)
                                except ValueError:
                                    continue
                
                # Read all drag files
                for drag_file in drag_files:
                    drag_path = os.path.join(dirpath, drag_file)
                    with open(drag_path) as f:
                        drag_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    drag_data.append(value)
                                except ValueError:
                                    continue
                
                # Apply angle of attack correction to transform forces
                # True lift = lift*cos(theta) - drag*sin(theta)
                # True drag = lift*sin(theta) + drag*cos(theta)
                cos_theta = np.cos(aoa_radians)
                sin_theta = np.sin(aoa_radians)
                
                true_lift_data = []
                true_drag_data = []
                
                # Ensure both arrays have the same length
                min_length = min(len(lift_data), len(drag_data))
                for i in range(min_length):
                    true_lift = lift_data[i] * cos_theta - drag_data[i] * sin_theta
                    true_drag = lift_data[i] * sin_theta + drag_data[i] * cos_theta
                    true_lift_data.append(true_lift)
                    true_drag_data.append(true_drag)
                
                # Store the corrected data
                data_by_config_aoa[(config, aoa)]['lift'].extend(true_lift_data)
                data_by_config_aoa[(config, aoa)]['drag'].extend(true_drag_data)
                
                # Store metadata
                data_by_config_aoa[(config, aoa)]['geometry'] = geometry
                data_by_config_aoa[(config, aoa)]['mesh'] = mesh
                data_by_config_aoa[(config, aoa)]['turbulence_model'] = turbulence_model
                data_by_config_aoa[(config, aoa)]['version'] = version
                data_by_config_aoa[(config, aoa)]['grid'] = grid
    
    return data_by_config_aoa


In [33]:
def compute_statistics(data):
    mean_val = np.mean(data)
    std_dev = np.std(data)
    # Calculate Coefficient of Variation (COV) as percentage
    cov = (std_dev / mean_val * 100) if mean_val != 0 else 0
    return mean_val, cov

## Enter Values Here

In [34]:
# ==================== USER INPUTS - MODIFY THESE ====================

# Input data directory
base_path = r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG"

# Output directory for results
output_dir = r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG"

# Number of iterations to use for statistics (if not using convergence analysis)
num_iterations = 150

# ==================== TURBULENCE MODEL COMPARISON ====================
# Specify which configurations to compare (base config without AoA, e.g., "4.3.1.2")
# Each turbulence model can have a different version, so specify one config per model
# Set to None or empty string to exclude that model from comparison

COMPARISON_CONFIGS = {
    'SST': '4.3.1.2',        # e.g., "4.3.1.2" - SST with Version 2
    'RNG': '4.3.2.1',        # e.g., "4.3.2.1" - RNG with Version 1  
    'RSM': '4.3.3.1',        # e.g., "4.3.3.1" - RSM with Version 1
    'k-epsilon': '',  # e.g., "4.3.4.2" - k-epsilon with Version 2
}

# ==================== CONVERGENCE ANALYSIS ====================
RUN_CONVERGENCE_ANALYSIS = True   # True = use optimized data, False = use fixed iterations
CONVERGENCE_MAX_TRIM = 0.8        # Max fraction of data to trim (0.8 = test up to 80% removal)
CONVERGENCE_NUM_TESTS = 20        # Number of trim points to test

# ==================== COEFFICIENT PARAMETERS ====================
SPAN = 0.85344          # meters - wingspan
CHORD = 0.1             # meters - chord length
AIR_DENSITY = 1.225     # kg/m^3 - air density at sea level
VELOCITY = 24.38        # m/s - freestream velocity

# Derived quantities (calculated automatically)
REFERENCE_AREA = SPAN * CHORD
DYNAMIC_PRESSURE = 0.5 * AIR_DENSITY * VELOCITY**2
Q_times_A = DYNAMIC_PRESSURE * REFERENCE_AREA

# ==================== CONFIGURATION PARSING ====================
CONFIG_EXTRACTION_METHOD = 'case_file'  # 'case_file' or 'folder_path'

# Position mapping: which position in config "4.3.1.2.NG.5" corresponds to which field
# Example: 4.3.1.2.NG.5 -> positions 0.1.2.3.4.5
POSITION_MAP = {
    'geometry': None,      # Position 0 = first number (4) - set to None if not used
    'mesh': 1,             # Position 1 = second number (3)
    'turbulence': 2,       # Position 2 = third number (1)
    'version': 3,          # Position 3 = fourth number (2)
    'grid': 4,             # Position 4 = fifth element (G or NG for Grid/No Grid)
    'aoa': 5               # Position 5 = sixth number (5)
}

# Value mappings: convert numeric codes to descriptive names
VALUE_MAPPINGS = {
    'geometry': {
        '1': 'Extended',
        '2': 'Extended, No Wingtip',
        '3': 'Minimized',
        '4': 'Baseline'
    },
    'mesh': {
        '1': 'Coarse Mesh',
        '2': 'Medium Mesh',
        '3': 'Standard',
        '4': 'NM Adjusted'
    },
    'turbulence': {
        '1': 'SST',
        '2': 'RNG',
        '3': 'RSM',
        '4': 'k-epsilon'
    },
    'version': {
        '1': 'Version 1',
        '2': 'Version 2',
        '3': 'Version 3',
        '4': 'Version 4'
    },
    'grid': {
        'G': 'Grid-Fins',
        'NG': 'No Grid-Fins'
    },
    'aoa': {
        0: 'AoA_0', 1: 'AoA_1', 2: 'AoA_2', 3: 'AoA_3', 4: 'AoA_4',
        5: 'AoA_5', 6: 'AoA_6', 7: 'AoA_7', 8: 'AoA_8', 9: 'AoA_9',
        10: 'AoA_10', 11: 'AoA_11', 12: 'AoA_12', 13: 'AoA_13', 14: 'AoA_14',
        15: 'AoA_15', 16: 'AoA_16', 17: 'AoA_17', 18: 'AoA_18', 19: 'AoA_19', 20: 'AoA_20'
    }
}

# ==================== DATA PROCESSING (DO NOT MODIFY BELOW) ====================

# Load all data
all_data = load_lift_drag_data(base_path)

# Create output directories
os.makedirs(output_dir, exist_ok=True)
raw_data_dir = os.path.join(output_dir, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

# Sort AoA values numerically (5, 10, 15, 20)
def extract_aoa_number(aoa_str):
    return int(aoa_str.split('_')[1])

# Export each dataset with a unique name based on config only (config already includes AoA)
for (config, aoa), data in all_data.items():
    # Use config as filename base since it already includes the AoA (e.g., "4.3.1.3.5")
    filename_base = config
    
    # Export lift data to raw_data folder
    lift_output = os.path.join(raw_data_dir, f"{filename_base}_lift.txt")
    with open(lift_output, 'w') as f:
        for value in data['lift']:
            f.write(f"{value}\n")
    
    # Export drag data to raw_data folder
    drag_output = os.path.join(raw_data_dir, f"{filename_base}_drag.txt")
    with open(drag_output, 'w') as f:
        for value in data['drag']:
            f.write(f"{value}\n")
    
    print(f"Exported: {filename_base}")

# Create summary statistics file using specified number of iterations
summary_file = os.path.join(output_dir, "SUMMARY_Statistics.txt")

with open(summary_file, 'w') as f:
    f.write("=" * 100 + "\n")
    f.write(f"SUMMARY STATISTICS - Last {num_iterations} Iterations\n")
    f.write("=" * 100 + "\n\n")
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations (or all if less than N)
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        
        # Write to file
        f.write(f"Configuration: {config}\n")
        f.write(f"Turbulence Model: {data['turbulence_model']}\n")
        f.write(f"Version: {data['version']}\n")
        f.write(f"Angle of Attack: {aoa}\n")
        f.write(f"Iterations used: {len(lift_last_n)}\n\n")
        
        f.write(f"LIFT:\n")
        f.write(f"  Average: {lift_mean:12.6f}\n")
        f.write(f"  COV (%): {lift_cov:12.6f}\n\n")
        
        f.write(f"DRAG:\n")
        f.write(f"  Average: {drag_mean:12.6f}\n")
        f.write(f"  COV (%): {drag_cov:12.6f}\n")
        f.write("-" * 100 + "\n\n")

print(f"\nSummary statistics written to: {summary_file}")

# Show available configurations for reference
print("\n" + "=" * 60)
print("AVAILABLE CONFIGURATIONS IN DATA:")
print("=" * 60)
unique_base_configs = set()
for (config, aoa), data in all_data.items():
    parts = config.split('.')
    if len(parts) >= 4:
        base_config = '.'.join(parts[:4])
        unique_base_configs.add((base_config, data['turbulence_model'], data['version']))

for cfg, turb, ver in sorted(unique_base_configs):
    print(f"  {cfg} = {turb}, {ver}")
print("=" * 60)

Exported: 3.1.1.NG.10
Exported: 3.1.1.NG.11
Exported: 3.1.1.NG.12
Exported: 3.1.1.NG.13
Exported: 3.1.1.NG.14
Exported: 3.1.1.NG.15
Exported: 3.1.1.NG.16
Exported: 3.1.1.NG.17
Exported: 3.1.1.NG.18
Exported: 3.1.1.NG.19
Exported: 3.1.1.NG.20
Exported: 3.1.1.NG.5
Exported: 3.1.1.NG.6
Exported: 3.1.1.NG.7
Exported: 3.1.1.NG.8
Exported: 3.1.1.NG.9

Summary statistics written to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\SUMMARY_Statistics.txt

AVAILABLE CONFIGURATIONS IN DATA:
  3.1.1.NG = SST, Version_NG


In [35]:
# ==================== CREATE EXCEL TABLES ====================

from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# Initialize convergence_results if not already defined (when RUN_CONVERGENCE_ANALYSIS is False)
if 'convergence_results' not in dir():
    convergence_results = {}

# Create Excel file with all data
excel_file = os.path.join(output_dir, "SUMMARY_Statistics.xlsx")

# Helper function to get optimized data for a configuration
def get_optimized_data(config, aoa, data, convergence_results):
    """Get optimized data if convergence analysis was run, otherwise use fixed iterations"""
    if RUN_CONVERGENCE_ANALYSIS and (config, aoa) in convergence_results:
        conv = convergence_results[(config, aoa)]
        lift_min_cov_idx = np.argmin(conv['lift']['cov'])
        drag_min_cov_idx = np.argmin(conv['drag']['cov'])
        
        optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
        optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
        optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
        
        lift_values = data['lift'][optimal_trim:]
        drag_values = data['drag'][optimal_trim:]
    else:
        # Use fixed iterations approach
        lift_values = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_values = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    return lift_values, drag_values

# Define turbulence model order for sorting
TURBULENCE_ORDER = {'SST': 1, 'RNG': 2, 'RSM': 3, 'k-epsilon': 4}

# Recreate config_groups for reference by turbulence model
config_groups = defaultdict(lambda: {'lift': {}, 'drag': {}})
for (config, aoa), data in all_data.items():
    config_parts = config.split('.')
    # Get turbulence number from position specified in POSITION_MAP
    turbulence_pos = POSITION_MAP['turbulence']
    turbulence_num = config_parts[turbulence_pos] if turbulence_pos is not None and len(config_parts) > turbulence_pos else "unknown"
    
    # Use optimized data
    lift_values, drag_values = get_optimized_data(config, aoa, data, convergence_results)
    
    config_groups[turbulence_num]['lift'][aoa] = lift_values
    config_groups[turbulence_num]['drag'][aoa] = drag_values

# ==================== CREATE DATA SUMMARY SHEET ====================
# Each unique configuration (e.g., "4.3.1.NG") gets its own section
# All AoAs for that configuration are listed together under the section header

# Define styles
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
group_header_font = Font(name='Calibri', size=12, bold=True, color='FFFFFF')
group_header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
data_alignment = Alignment(horizontal='center', vertical='center')
border_style = Border(
    left=Side(style='thin', color='000000'),
    right=Side(style='thin', color='000000'),
    top=Side(style='thin', color='000000'),
    bottom=Side(style='thin', color='000000')
)
row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')

# Group data by base configuration (the config string like "4.3.1.NG")
# AoA is the LAST part of the config string, so we need to strip it off
base_config_data = defaultdict(list)
for (config, aoa), data in all_data.items():
    # Extract base config by removing the AoA (last part after the last dot)
    config_parts = config.split('.')
    # Remove the last part (AoA) to get base config like "4.3.1.NG"
    if len(config_parts) > 1:
        base_config = '.'.join(config_parts[:-1])
    else:
        base_config = config
    
    # Use optimized data
    lift_values, drag_values = get_optimized_data(config, aoa, data, convergence_results)
    
    # Calculate statistics
    lift_mean = np.mean(lift_values) if lift_values else 0
    lift_std = np.std(lift_values) if lift_values else 0
    lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
    
    drag_mean = np.mean(drag_values) if drag_values else 0
    drag_std = np.std(drag_values) if drag_values else 0
    drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
    
    base_config_data[base_config].append({
        'aoa': aoa,
        'aoa_num': extract_aoa_number(aoa),
        'turbulence_model': data['turbulence_model'],
        'num_points': len(lift_values),
        'lift_mean': lift_mean,
        'lift_cov': lift_cov,
        'drag_mean': drag_mean,
        'drag_cov': drag_cov
    })

# Sort each group by AoA
for base_config in base_config_data:
    base_config_data[base_config].sort(key=lambda x: x['aoa_num'])

# Sort base configs by turbulence model order, then by config name
def get_base_config_sort_key(base_config):
    if base_config in base_config_data and base_config_data[base_config]:
        turb_model = base_config_data[base_config][0]['turbulence_model']
        return (TURBULENCE_ORDER.get(turb_model, 99), base_config)
    return (99, base_config)

sorted_base_configs = sorted(base_config_data.keys(), key=get_base_config_sort_key)

# Create Excel workbook manually for full control
from openpyxl import Workbook
wb = Workbook()
ws = wb.active
ws.title = 'Data Summary'

# Column headers (including Turbulence Model column)
columns = ['Turbulence Model', 'Angle of Attack', 'Lift Mean (N)', 'Lift COV (%)', 'Drag Mean (N)', 'Drag COV (%)', 'Num Points']

current_row = 1

for base_config in sorted_base_configs:
    data_rows = base_config_data[base_config]
    if not data_rows:
        continue
    
    # Write section header (base configuration name)
    ws.cell(row=current_row, column=1, value=base_config)
    # Merge cells for section header
    ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(columns))
    header_cell = ws.cell(row=current_row, column=1)
    header_cell.font = group_header_font
    header_cell.fill = group_header_fill
    header_cell.alignment = Alignment(horizontal='center', vertical='center')
    header_cell.border = border_style
    ws.row_dimensions[current_row].height = 25
    current_row += 1
    
    # Write column headers
    for col_idx, col_name in enumerate(columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    ws.row_dimensions[current_row].height = 25
    current_row += 1
    
    # Write data rows - all AoAs for this configuration
    for row_idx, row_data in enumerate(data_rows):
        fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
        
        values = [
            row_data['turbulence_model'],
            row_data['aoa'],
            f"{row_data['lift_mean']:.1f}",
            f"{row_data['lift_cov']:.1f}",
            f"{row_data['drag_mean']:.1f}",
            f"{row_data['drag_cov']:.1f}",
            row_data['num_points']
        ]
        
        for col_idx, val in enumerate(values, 1):
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
        
        current_row += 1
    
    # Add blank row between sections
    current_row += 1

# Autofit column widths
for col_idx in range(1, len(columns) + 1):
    max_length = 0
    column_letter = get_column_letter(col_idx)
    for row in ws.iter_rows(min_col=col_idx, max_col=col_idx):
        for cell in row:
            try:
                if cell.value and len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
    adjusted_width = min(max_length + 2, 25)
    ws.column_dimensions[column_letter].width = adjusted_width

wb.save(excel_file)

print(f"✓ Excel file created: {excel_file}")
print(f"✓ Sheet: Data Summary")
print(f"✓ Each configuration (e.g., 4.3.1.NG) has its own section")
print(f"✓ All AoAs for each configuration grouped under its section header")
print(f"✓ Columns: AoA | Lift Mean | Lift COV | Drag Mean | Drag COV | Num Points")
print(f"✓ Sorting: Turbulence Model (SST → RNG → RSM → k-epsilon), then AoA")

✓ Excel file created: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\SUMMARY_Statistics.xlsx
✓ Sheet: Data Summary
✓ Each configuration (e.g., 4.3.1.NG) has its own section
✓ All AoAs for each configuration grouped under its section header
✓ Columns: AoA | Lift Mean | Lift COV | Drag Mean | Drag COV | Num Points
✓ Sorting: Turbulence Model (SST → RNG → RSM → k-epsilon), then AoA


In [36]:
# ==================== CREATE TURBULENCE MODEL COMPARISON TABLE ====================
# Uses COMPARISON_CONFIGS from user inputs to specify which configs to compare
# Condensed format: Two sections (Lift and Drag) with simple headers

import time
from openpyxl.utils import get_column_letter

print("=" * 70)
print("TURBULENCE MODEL COMPARISON")
print("=" * 70)

# Show which configs are being compared - filter out None, empty strings, and models with no data
print("\nConfigurations being compared:")
active_configs = {k: v for k, v in COMPARISON_CONFIGS.items() if v and str(v).strip()}
for model, config in active_configs.items():
    print(f"  {model}: {config}")

if not active_configs:
    print("⚠ No configurations specified in COMPARISON_CONFIGS!")
    print("  Please set config values in the User Inputs cell.")
else:
    # Get all unique AoAs from the specified configs
    all_aoas = set()
    for (config, aoa), data in all_data.items():
        parts = config.split('.')
        if len(parts) >= 4:
            base_config = '.'.join(parts[:4])
            if base_config in active_configs.values():
                all_aoas.add(aoa)
    
    # Sort AoAs numerically
    sorted_aoas = sorted(all_aoas, key=extract_aoa_number)
    print(f"\nAoAs found in specified configs: {sorted_aoas}")
    
    # Sort models by order - only include models that are active
    model_order = {'SST': 1, 'RNG': 2, 'RSM': 3, 'k-epsilon': 4}
    sorted_models = sorted(active_configs.keys(), key=lambda x: model_order.get(x, 99))
    print(f"Models included: {sorted_models}")
    
    # Collect data for all AoAs and models
    lift_data = {}  # {aoa: {model: mean}}
    drag_data = {}  # {aoa: {model: mean}}
    lift_cov_data = {}  # {aoa: {model: cov}}
    drag_cov_data = {}  # {aoa: {model: cov}}
    
    for aoa in sorted_aoas:
        lift_data[aoa] = {}
        drag_data[aoa] = {}
        lift_cov_data[aoa] = {}
        drag_cov_data[aoa] = {}
        
        for model in sorted_models:
            base_config = active_configs[model]
            model_lift_values = []
            model_drag_values = []
            
            # Find the specific config for this model and AoA
            for (config, config_aoa), data in all_data.items():
                if config_aoa == aoa:
                    parts = config.split('.')
                    if len(parts) >= 4:
                        this_base = '.'.join(parts[:4])
                        if this_base == base_config:
                            lift_values, drag_values = get_optimized_data(config, aoa, data, convergence_results)
                            if lift_values:
                                model_lift_values.extend(lift_values)
                            if drag_values:
                                model_drag_values.extend(drag_values)
            
            lift_data[aoa][model] = np.mean(model_lift_values) if model_lift_values else None
            drag_data[aoa][model] = np.mean(model_drag_values) if model_drag_values else None
            
            # Calculate COV for each model
            if model_lift_values:
                lift_mean = np.mean(model_lift_values)
                lift_std = np.std(model_lift_values)
                lift_cov_data[aoa][model] = (lift_std / lift_mean * 100) if lift_mean != 0 else None
            else:
                lift_cov_data[aoa][model] = None
                
            if model_drag_values:
                drag_mean = np.mean(model_drag_values)
                drag_std = np.std(model_drag_values)
                drag_cov_data[aoa][model] = (drag_std / drag_mean * 100) if drag_mean != 0 else None
            else:
                drag_cov_data[aoa][model] = None
    
    # ==================== CREATE LIFT COMPARISON TABLE ====================
    lift_rows = []
    for aoa in sorted_aoas:
        row = {'AoA': aoa}
        sst_lift = lift_data[aoa].get('SST')
        
        for model in sorted_models:
            val = lift_data[aoa].get(model)
            if val is not None:
                row[f'{model} (N)'] = round(val, 1)
                # Add %Diff column for non-SST models
                if model != 'SST' and sst_lift is not None and sst_lift != 0:
                    pct_diff = ((val - sst_lift) / sst_lift * 100)
                    row[f'{model} %Diff'] = round(pct_diff, 1)
                elif model != 'SST':
                    row[f'{model} %Diff'] = 'N/A'
            else:
                row[f'{model} (N)'] = 'N/A'
                if model != 'SST':
                    row[f'{model} %Diff'] = 'N/A'
        
        lift_rows.append(row)
    
    lift_df = pd.DataFrame(lift_rows)
    
    # ==================== CREATE DRAG COMPARISON TABLE ====================
    drag_rows = []
    for aoa in sorted_aoas:
        row = {'AoA': aoa}
        sst_drag = drag_data[aoa].get('SST')
        
        for model in sorted_models:
            val = drag_data[aoa].get(model)
            if val is not None:
                row[f'{model} (N)'] = round(val, 1)
                # Add %Diff column for non-SST models
                if model != 'SST' and sst_drag is not None and sst_drag != 0:
                    pct_diff = ((val - sst_drag) / sst_drag * 100)
                    row[f'{model} %Diff'] = round(pct_diff, 1)
                elif model != 'SST':
                    row[f'{model} %Diff'] = 'N/A'
            else:
                row[f'{model} (N)'] = 'N/A'
                if model != 'SST':
                    row[f'{model} %Diff'] = 'N/A'
        
        drag_rows.append(row)
    
    drag_df = pd.DataFrame(drag_rows)
    
    print(f"\nLift table: {len(lift_df)} rows, columns: {list(lift_df.columns)}")
    print(f"Drag table: {len(drag_df)} rows, columns: {list(drag_df.columns)}")
    
    # ==================== CREATE LIFT COV COMPARISON TABLE ====================
    lift_cov_rows = []
    for aoa in sorted_aoas:
        row = {'AoA': aoa}
        sst_lift_cov = lift_cov_data[aoa].get('SST')
        
        for model in sorted_models:
            val = lift_cov_data[aoa].get(model)
            if val is not None:
                row[f'{model} COV (%)'] = round(val, 1)
                # Add %Diff column for non-SST models
                if model != 'SST' and sst_lift_cov is not None and sst_lift_cov != 0:
                    pct_diff = ((val - sst_lift_cov) / sst_lift_cov * 100)
                    row[f'{model} COV %Diff'] = round(pct_diff, 1)
                elif model != 'SST':
                    row[f'{model} COV %Diff'] = 'N/A'
            else:
                row[f'{model} COV (%)'] = 'N/A'
                if model != 'SST':
                    row[f'{model} COV %Diff'] = 'N/A'
        
        lift_cov_rows.append(row)
    
    lift_cov_df = pd.DataFrame(lift_cov_rows)
    
    # ==================== CREATE DRAG COV COMPARISON TABLE ====================
    drag_cov_rows = []
    for aoa in sorted_aoas:
        row = {'AoA': aoa}
        sst_drag_cov = drag_cov_data[aoa].get('SST')
        
        for model in sorted_models:
            val = drag_cov_data[aoa].get(model)
            if val is not None:
                row[f'{model} COV (%)'] = round(val, 1)
                # Add %Diff column for non-SST models
                if model != 'SST' and sst_drag_cov is not None and sst_drag_cov != 0:
                    pct_diff = ((val - sst_drag_cov) / sst_drag_cov * 100)
                    row[f'{model} COV %Diff'] = round(pct_diff, 1)
                elif model != 'SST':
                    row[f'{model} COV %Diff'] = 'N/A'
            else:
                row[f'{model} COV (%)'] = 'N/A'
                if model != 'SST':
                    row[f'{model} COV %Diff'] = 'N/A'
        
        drag_cov_rows.append(row)
    
    drag_cov_df = pd.DataFrame(drag_cov_rows)
    
    print(f"Lift COV table: {len(lift_cov_df)} rows, columns: {list(lift_cov_df.columns)}")
    print(f"Drag COV table: {len(drag_cov_df)} rows, columns: {list(drag_cov_df.columns)}")
    
    # ==================== WRITE TO EXCEL ====================
    # Create a new workbook for this sheet to avoid conflicts
    # Row layout:
    # 1: LIFT COMPARISON title
    # 2: Lift headers
    # 3-N: Lift data
    # N+2: (blank)
    # N+3: DRAG COMPARISON title
    # N+4: Drag headers
    # N+5+: Drag data
    
    # Delete existing sheet if it exists and create fresh
    try:
        wb_existing = load_workbook(excel_file)
        if 'Turbulence_Comparison' in wb_existing.sheetnames:
            del wb_existing['Turbulence_Comparison']
            wb_existing.save(excel_file)
        wb_existing.close()
    except:
        pass
    
    time.sleep(0.3)
    
    # Now create the sheet manually with openpyxl for full control
    wb = load_workbook(excel_file)
    ws = wb.create_sheet('Turbulence_Comparison')
    
    # Define styles
    header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
    header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
    section_font = Font(name='Calibri', size=12, bold=True, color='366092')
    header_alignment = Alignment(horizontal='center', vertical='center')
    data_alignment = Alignment(horizontal='center', vertical='center')
    border_style = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
    row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')
    diff_font = Font(name='Calibri', size=11, color='4169E1')  # Blue for %Diff
    
    current_row = 1
    
    # ===== LIFT SECTION =====
    # Title row
    ws.cell(row=current_row, column=1, value="LIFT COMPARISON (N)")
    ws.cell(row=current_row, column=1).font = section_font
    current_row += 1
    
    # Header row
    lift_header_row = current_row
    for col_idx, col_name in enumerate(lift_df.columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    current_row += 1
    
    # Data rows
    lift_data_start = current_row
    for row_idx, row_data in lift_df.iterrows():
        fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
        for col_idx, col_name in enumerate(lift_df.columns, 1):
            val = row_data[col_name]
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
            if '%Diff' in col_name:
                cell.font = diff_font
        current_row += 1
    
    lift_data_end = current_row - 1
    current_row += 1  # Blank row
    
    # ===== DRAG SECTION =====
    # Title row
    ws.cell(row=current_row, column=1, value="DRAG COMPARISON (N)")
    ws.cell(row=current_row, column=1).font = section_font
    current_row += 1
    
    # Header row
    drag_header_row = current_row
    for col_idx, col_name in enumerate(drag_df.columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    current_row += 1
    
    # Data rows
    drag_data_start = current_row
    for row_idx, row_data in drag_df.iterrows():
        fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
        for col_idx, col_name in enumerate(drag_df.columns, 1):
            val = row_data[col_name]
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
            if '%Diff' in col_name:
                cell.font = diff_font
        current_row += 1
    
    drag_data_end = current_row - 1
    current_row += 2  # Blank rows
    
    # ===== LIFT COV SECTION =====
    # Title row
    ws.cell(row=current_row, column=1, value="LIFT COV COMPARISON (%)")
    ws.cell(row=current_row, column=1).font = section_font
    current_row += 1
    
    # Header row
    lift_cov_header_row = current_row
    for col_idx, col_name in enumerate(lift_cov_df.columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    current_row += 1
    
    # Data rows
    lift_cov_data_start = current_row
    for row_idx, row_data in lift_cov_df.iterrows():
        fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
        for col_idx, col_name in enumerate(lift_cov_df.columns, 1):
            val = row_data[col_name]
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
            if '%Diff' in col_name:
                cell.font = diff_font
        current_row += 1
    
    lift_cov_data_end = current_row - 1
    current_row += 1  # Blank row
    
    # ===== DRAG COV SECTION =====
    # Title row
    ws.cell(row=current_row, column=1, value="DRAG COV COMPARISON (%)")
    ws.cell(row=current_row, column=1).font = section_font
    current_row += 1
    
    # Header row
    drag_cov_header_row = current_row
    for col_idx, col_name in enumerate(drag_cov_df.columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    current_row += 1
    
    # Data rows
    drag_cov_data_start = current_row
    for row_idx, row_data in drag_cov_df.iterrows():
        fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
        for col_idx, col_name in enumerate(drag_cov_df.columns, 1):
            val = row_data[col_name]
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
            if '%Diff' in col_name:
                cell.font = diff_font
        current_row += 1
    
    drag_cov_data_end = current_row - 1
    current_row += 2  # Blank rows
    
    # Config reference note
    ws.cell(row=current_row, column=1, value="Configurations compared:")
    ws.cell(row=current_row, column=1).font = Font(name='Calibri', size=10, italic=True)
    current_row += 1
    
    for model, config in active_configs.items():
        ws.cell(row=current_row, column=1, value=f"  {model}: {config}")
        ws.cell(row=current_row, column=1).font = Font(name='Calibri', size=10, italic=True)
        current_row += 1
    
    # Autofit column widths - use the max of all table columns
    max_cols = max(len(lift_df.columns), len(drag_df.columns), len(lift_cov_df.columns), len(drag_cov_df.columns))
    for col_idx in range(1, max_cols + 1):
        max_length = 0
        column_letter = get_column_letter(col_idx)
        for row in ws.iter_rows(min_col=col_idx, max_col=col_idx):
            for cell in row:
                try:
                    if cell.value and len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
        adjusted_width = min(max_length + 2, 18)
        ws.column_dimensions[column_letter].width = adjusted_width
    
    wb.save(excel_file)
    wb.close()
    
    print(f"\n✓ Turbulence Comparison table created")
    print(f"✓ LIFT section: rows {lift_header_row}-{lift_data_end}")
    print(f"✓ DRAG section: rows {drag_header_row}-{drag_data_end}")
    print(f"✓ LIFT COV section: rows {lift_cov_header_row}-{lift_cov_data_end}")
    print(f"✓ DRAG COV section: rows {drag_cov_header_row}-{drag_cov_data_end}")
    print(f"✓ Columns: AoA | Model values | %Diff vs SST")

TURBULENCE MODEL COMPARISON

Configurations being compared:
  SST: 4.3.1.2
  RNG: 4.3.2.1
  RSM: 4.3.3.1

AoAs found in specified configs: []
Models included: ['SST', 'RNG', 'RSM']

Lift table: 0 rows, columns: []
Drag table: 0 rows, columns: []
Lift COV table: 0 rows, columns: []
Drag COV table: 0 rows, columns: []

✓ Turbulence Comparison table created
✓ LIFT section: rows 2-2
✓ DRAG section: rows 5-5
✓ LIFT COV section: rows 9-9
✓ DRAG COV section: rows 12-12
✓ Columns: AoA | Model values | %Diff vs SST

✓ Turbulence Comparison table created
✓ LIFT section: rows 2-2
✓ DRAG section: rows 5-5
✓ LIFT COV section: rows 9-9
✓ DRAG COV section: rows 12-12
✓ Columns: AoA | Model values | %Diff vs SST


## Convergence Analysis (Optional)

In [37]:
# ==================== CONVERGENCE ANALYSIS TOOL ====================
# Inspired by GUI_Plotter_v12.txt methodology
# Tests different amounts of initial data removal to find optimal convergence window

def analyze_convergence(data_array, min_trim=0, max_trim=0.5, num_tests=10):
    """
    Analyze how statistics change when trimming different amounts of initial data.
    
    Parameters:
    - data_array: numpy array of lift or drag values
    - min_trim: minimum fraction of data to trim from beginning (0 to 1)
    - max_trim: maximum fraction of data to trim from beginning (0 to 1)
    - num_tests: number of trim amounts to test
    
    Returns:
    - Dictionary with trim fractions, means, std devs, and COVs
    """
    total_iterations = len(data_array)
    trim_fractions = np.linspace(min_trim, max_trim, num_tests)
    
    results = {
        'iterations_removed': [],
        'iterations_used': [],
        'mean': [],
        'std_dev': [],
        'cov': []
    }
    
    for trim_frac in trim_fractions:
        iterations_to_remove = int(total_iterations * trim_frac)
        trimmed_data = data_array[iterations_to_remove:]
        
        if len(trimmed_data) > 0:
            mean_val = np.mean(trimmed_data)
            std_val = np.std(trimmed_data)
            cov_val = (std_val / mean_val * 100) if mean_val != 0 else 0
            
            results['iterations_removed'].append(iterations_to_remove)
            results['iterations_used'].append(len(trimmed_data))
            results['mean'].append(mean_val)
            results['std_dev'].append(std_val)
            results['cov'].append(cov_val)
    
    return results

def plot_convergence_analysis(config, aoa, lift_data, drag_data, output_dir):
    """
    Create convergence analysis plots showing how statistics change with data trimming.
    """
    # Analyze both lift and drag using configured parameters
    lift_results = analyze_convergence(np.array(lift_data), min_trim=0, max_trim=CONVERGENCE_MAX_TRIM, num_tests=CONVERGENCE_NUM_TESTS)
    drag_results = analyze_convergence(np.array(drag_data), min_trim=0, max_trim=CONVERGENCE_MAX_TRIM, num_tests=CONVERGENCE_NUM_TESTS)
    
    # Create figure with subplots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Lift Mean vs Iterations Removed
    ax1.plot(lift_results['iterations_removed'], lift_results['mean'], 'o-', linewidth=2, markersize=8, color='#1f77b4')
    ax1.set_xlabel('Iterations Removed from Start')
    ax1.set_ylabel('Lift Mean (N)')
    ax1.set_title(f'Lift Mean Convergence\n{config} - {aoa}', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Lift COV vs Iterations Removed
    ax2.plot(lift_results['iterations_removed'], lift_results['cov'], 'o-', linewidth=2, markersize=8, color='#ff7f0e')
    ax2.set_xlabel('Iterations Removed from Start')
    ax2.set_ylabel('Lift COV (%)')
    ax2.set_title(f'Lift COV Convergence\n{config} - {aoa}', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Highlight minimum COV point for lift
    min_cov_idx = np.argmin(lift_results['cov'])
    ax2.axvline(x=lift_results['iterations_removed'][min_cov_idx], color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax2.text(lift_results['iterations_removed'][min_cov_idx], max(lift_results['cov']), 
             f"  Min COV\n  Remove: {lift_results['iterations_removed'][min_cov_idx]}\n  Use: {lift_results['iterations_used'][min_cov_idx]}", 
             color='red', fontweight='bold', fontsize=9)
    
    # Plot 3: Drag Mean vs Iterations Removed
    ax3.plot(drag_results['iterations_removed'], drag_results['mean'], 'o-', linewidth=2, markersize=8, color='#2ca02c')
    ax3.set_xlabel('Iterations Removed from Start')
    ax3.set_ylabel('Drag Mean (N)')
    ax3.set_title(f'Drag Mean Convergence\n{config} - {aoa}', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Drag COV vs Iterations Removed
    ax4.plot(drag_results['iterations_removed'], drag_results['cov'], 'o-', linewidth=2, markersize=8, color='#d62728')
    ax4.set_xlabel('Iterations Removed from Start')
    ax4.set_ylabel('Drag COV (%)')
    ax4.set_title(f'Drag COV Convergence\n{config} - {aoa}', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    # Highlight minimum COV point for drag
    min_cov_idx = np.argmin(drag_results['cov'])
    ax4.axvline(x=drag_results['iterations_removed'][min_cov_idx], color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax4.text(drag_results['iterations_removed'][min_cov_idx], max(drag_results['cov']), 
             f"  Min COV\n  Remove: {drag_results['iterations_removed'][min_cov_idx]}\n  Use: {drag_results['iterations_used'][min_cov_idx]}", 
             color='red', fontweight='bold', fontsize=9)
    
    plt.tight_layout()
    
    # Save convergence analysis plot
    convergence_dir = os.path.join(output_dir, "convergence_analysis")
    os.makedirs(convergence_dir, exist_ok=True)
    
    plot_file = os.path.join(convergence_dir, f"convergence_{config}.png")
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    return lift_results, drag_results, plot_file

# ==================== RUN CONVERGENCE ANALYSIS ====================
# Check if convergence analysis is enabled in configuration section above

if RUN_CONVERGENCE_ANALYSIS:
    print("\n" + "=" * 80)
    print("RUNNING CONVERGENCE ANALYSIS")
    print("=" * 80)
    
    # Create convergence analysis directory
    convergence_dir = os.path.join(output_dir, "convergence_analysis")
    os.makedirs(convergence_dir, exist_ok=True)
    
    # Run convergence analysis on all configurations
    convergence_results = {}
    
    # Create text file for convergence data
    convergence_text_file = os.path.join(convergence_dir, "Convergence_Analysis_Results.txt")
    
    with open(convergence_text_file, 'w') as f:
        f.write("=" * 120 + "\n")
        f.write("CONVERGENCE ANALYSIS RESULTS\n")
        f.write("=" * 120 + "\n\n")
        
        for (config, aoa), data in all_data.items():
            print(f"\nAnalyzing: {config} - {aoa}")
            
            lift_results, drag_results, plot_path = plot_convergence_analysis(
                config, aoa,
                data['lift'],
                data['drag'],
                output_dir
            )
            
            convergence_results[(config, aoa)] = {
                'lift': lift_results,
                'drag': drag_results,
                'plot': plot_path
            }
            
            # Print optimization recommendations
            lift_min_cov_idx = np.argmin(lift_results['cov'])
            drag_min_cov_idx = np.argmin(drag_results['cov'])
            
            print(f"  ✓ Plot saved: {plot_path}")
            print(f"  ✓ Lift - Min COV at {lift_results['iterations_removed'][lift_min_cov_idx]} iterations removed (use {lift_results['iterations_used'][lift_min_cov_idx]})")
            print(f"  ✓ Drag - Min COV at {drag_results['iterations_removed'][drag_min_cov_idx]} iterations removed (use {drag_results['iterations_used'][drag_min_cov_idx]})")
            
            # Write to text file
            f.write(f"Configuration: {config}\n")
            f.write(f"Angle of Attack: {aoa}\n")
            f.write(f"Total Iterations: {len(data['lift'])}\n")
            f.write("-" * 120 + "\n\n")
            
            # Write LIFT table
            f.write("LIFT CONVERGENCE:\n")
            f.write(f"{'Iterations_Removed':<20} {'Iterations_Used':<20} {'Mean':<15} {'StdDev':<15} {'COV(%)':<10}\n")
            f.write("-" * 120 + "\n")
            for i in range(len(lift_results['iterations_removed'])):
                marker = " <-- MIN COV" if i == lift_min_cov_idx else ""
                f.write(f"{lift_results['iterations_removed'][i]:<20} "
                       f"{lift_results['iterations_used'][i]:<20} "
                       f"{lift_results['mean'][i]:<15.6f} "
                       f"{lift_results['std_dev'][i]:<15.6f} "
                       f"{lift_results['cov'][i]:<10.2f}{marker}\n")
            
            f.write("\n")
            
            # Write DRAG table
            f.write("DRAG CONVERGENCE:\n")
            f.write(f"{'Iterations_Removed':<20} {'Iterations_Used':<20} {'Mean':<15} {'StdDev':<15} {'COV(%)':<10}\n")
            f.write("-" * 120 + "\n")
            for i in range(len(drag_results['iterations_removed'])):
                marker = " <-- MIN COV" if i == drag_min_cov_idx else ""
                f.write(f"{drag_results['iterations_removed'][i]:<20} "
                       f"{drag_results['iterations_used'][i]:<20} "
                       f"{drag_results['mean'][i]:<15.6f} "
                       f"{drag_results['std_dev'][i]:<15.6f} "
                       f"{drag_results['cov'][i]:<10.2f}{marker}\n")
            
            f.write("\n" + "=" * 120 + "\n\n")
    
    print("\n" + "=" * 80)
    print("CONVERGENCE ANALYSIS COMPLETE")
    print(f"✓ Total configurations analyzed: {len(convergence_results)}")
    print(f"✓ Plots saved to: {os.path.join(output_dir, 'convergence_analysis')}")
    print(f"✓ Text file saved: {convergence_text_file}")
    print("=" * 80)
    
    # ==================== OPTIMIZED STATISTICS ====================
    # Apply convergence findings to recalculate statistics with optimal trim points
    print("\n" + "=" * 80)
    print("APPLYING OPTIMIZED CONVERGENCE SETTINGS")
    print("=" * 80)
    
    # Define turbulence model order for sorting
    TURBULENCE_ORDER = {'SST': 1, 'RNG': 2, 'RSM': 3, 'k-epsilon': 4}
    
    optimized_summary = []
    
    # Sort by turbulence model, then config, then AoA
    sorted_items = sorted(all_data.items(), 
                          key=lambda x: (TURBULENCE_ORDER.get(x[1]['turbulence_model'], 99),
                                        x[0][0],
                                        extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_items:
        if (config, aoa) in convergence_results:
            conv = convergence_results[(config, aoa)]
            
            # Find optimal trim points (minimum COV)
            lift_min_cov_idx = np.argmin(conv['lift']['cov'])
            drag_min_cov_idx = np.argmin(conv['drag']['cov'])
            
            optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
            optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
            
            # Use the more conservative (higher) trim value for both
            optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
            
            # Apply optimal trim
            lift_optimized = data['lift'][optimal_trim:]
            drag_optimized = data['drag'][optimal_trim:]
            
            # Calculate optimized statistics
            lift_mean_opt = np.mean(lift_optimized) if lift_optimized else 0
            lift_std_opt = np.std(lift_optimized) if lift_optimized else 0
            lift_cov_opt = (lift_std_opt / lift_mean_opt * 100) if lift_mean_opt != 0 else 0
            
            drag_mean_opt = np.mean(drag_optimized) if drag_optimized else 0
            drag_std_opt = np.std(drag_optimized) if drag_optimized else 0
            drag_cov_opt = (drag_std_opt / drag_mean_opt * 100) if drag_mean_opt != 0 else 0
            
            # Compare with original fixed-iteration statistics
            lift_original = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
            drag_original = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
            
            lift_mean_orig = np.mean(lift_original) if lift_original else 0
            lift_cov_orig = (np.std(lift_original) / lift_mean_orig * 100) if lift_mean_orig != 0 else 0
            
            drag_mean_orig = np.mean(drag_original) if drag_original else 0
            drag_cov_orig = (np.std(drag_original) / drag_mean_orig * 100) if drag_mean_orig != 0 else 0
            
            # Calculate improvements
            lift_cov_improvement = lift_cov_orig - lift_cov_opt
            drag_cov_improvement = drag_cov_orig - drag_cov_opt
            
            optimized_summary.append({
                'Turbulence Model': data['turbulence_model'],
                'Configuration': config,
                'AoA': aoa,
                'Original Iterations': len(lift_original),
                'Optimal Trim': optimal_trim,
                'Optimized Iterations': len(lift_optimized),
                'Lift Mean (Orig)': f"{lift_mean_orig:.3f}",
                'Lift Mean (Opt)': f"{lift_mean_opt:.3f}",
                'Lift COV (Orig)': f"{lift_cov_orig:.2f}%",
                'Lift COV (Opt)': f"{lift_cov_opt:.2f}%",
                'Lift COV Δ': f"{lift_cov_improvement:+.2f}%",
                'Drag Mean (Orig)': f"{drag_mean_orig:.3f}",
                'Drag Mean (Opt)': f"{drag_mean_opt:.3f}",
                'Drag COV (Orig)': f"{drag_cov_orig:.2f}%",
                'Drag COV (Opt)': f"{drag_cov_opt:.2f}%",
                'Drag COV Δ': f"{drag_cov_improvement:+.2f}%"
            })
            
            print(f"\n{config} - {aoa}:")
            print(f"  Optimal trim: {optimal_trim} iterations")
            print(f"  Lift COV: {lift_cov_orig:.2f}% → {lift_cov_opt:.2f}% (Δ {lift_cov_improvement:+.2f}%)")
            print(f"  Drag COV: {drag_cov_orig:.2f}% → {drag_cov_opt:.2f}% (Δ {drag_cov_improvement:+.2f}%)")
    
    # Save optimized statistics to Excel
    optimized_df = pd.DataFrame(optimized_summary)
    with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        optimized_df.to_excel(writer, sheet_name='Optimized_Statistics', index=False)
    
    # Apply formatting
    import time
    time.sleep(0.5)
    wb = load_workbook(excel_file)
    ws = wb['Optimized_Statistics']
    
    # Apply same professional formatting
    for column in ws.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 25)
        ws.column_dimensions[column_letter].width = adjusted_width
    
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
        for cell in row:
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
    
    ws.row_dimensions[1].height = 30
    wb.save(excel_file)
    wb.close()
    
    print("\n" + "=" * 80)
    print("OPTIMIZATION COMPLETE")
    print(f"✓ Optimized statistics saved to Excel sheet: 'Optimized_Statistics'")
    print(f"✓ Average COV improvements calculated for all configurations")
    print("=" * 80)
    
else:
    print("\n✓ Convergence analysis skipped (set RUN_CONVERGENCE_ANALYSIS = True to enable)")


RUNNING CONVERGENCE ANALYSIS

Analyzing: 3.1.1.NG.10 - AoA_10
  ✓ Plot saved: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_analysis\convergence_3.1.1.NG.10.png
  ✓ Lift - Min COV at 960 iterations removed (use 240)
  ✓ Drag - Min COV at 909 iterations removed (use 291)

Analyzing: 3.1.1.NG.11 - AoA_11
  ✓ Plot saved: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_analysis\convergence_3.1.1.NG.10.png
  ✓ Lift - Min COV at 960 iterations removed (use 240)
  ✓ Drag - Min COV at 909 iterations removed (use 291)

Analyzing: 3.1.1.NG.11 - AoA_11
  ✓ Plot saved: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_analysis\convergence_3.1.1.NG.11.png
  ✓ Lift - Min COV at 353 iterations removed (use 847)
  ✓ Drag - Min COV at 353 iterations removed (use 847)

Analyzing: 3.1.1.NG.12 - AoA_1

## Coefficient Calculations and AoA Graphs

In [38]:
# ==================== CALCULATE COEFFICIENTS ====================
# Calculate C_L and C_D for each configuration

coefficient_data = {}

for (config, aoa), data in all_data.items():
    # Get optimized data if convergence analysis was run, otherwise use fixed iterations
    if RUN_CONVERGENCE_ANALYSIS and (config, aoa) in convergence_results:
        conv = convergence_results[(config, aoa)]
        lift_min_cov_idx = np.argmin(conv['lift']['cov'])
        drag_min_cov_idx = np.argmin(conv['drag']['cov'])
        
        optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
        optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
        optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
        
        lift_values = data['lift'][optimal_trim:]
        drag_values = data['drag'][optimal_trim:]
    else:
        # Use fixed iterations approach
        lift_values = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_values = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate mean forces
    lift_mean = np.mean(lift_values) if lift_values else 0
    drag_mean = np.mean(drag_values) if drag_values else 0
    
    # Calculate coefficients
    C_L = lift_mean / Q_times_A if Q_times_A != 0 else 0
    C_D = drag_mean / Q_times_A if Q_times_A != 0 else 0
    
    # Calculate standard deviations and COV for coefficients
    lift_std = np.std(lift_values) if lift_values else 0
    drag_std = np.std(drag_values) if drag_values else 0
    
    C_L_std = lift_std / Q_times_A if Q_times_A != 0 else 0
    C_D_std = drag_std / Q_times_A if Q_times_A != 0 else 0
    
    C_L_cov = (C_L_std / C_L * 100) if C_L != 0 else 0
    C_D_cov = (C_D_std / C_D * 100) if C_D != 0 else 0
    
    # Extract AoA number
    aoa_degrees = extract_aoa_number(aoa)
    
    # Extract config parts for filtering
    config_parts = config.split('.')
    
    # Get version number from config
    version_num = config_parts[POSITION_MAP['version']] if len(config_parts) > POSITION_MAP['version'] else 'unknown'
    
    # Store coefficient data
    coefficient_data[(config, aoa)] = {
        'config': config,
        'config_parts': config_parts,
        'version_num': version_num,
        'aoa_degrees': aoa_degrees,
        'C_L': C_L,
        'C_D': C_D,
        'C_L_std': C_L_std,
        'C_D_std': C_D_std,
        'C_L_cov': C_L_cov,
        'C_D_cov': C_D_cov,
        'turbulence_model': data['turbulence_model'],
        'geometry': data['geometry'],
        'mesh': data['mesh'],
        'grid': data['grid']
    }

print(f"\n✓ Coefficients calculated for {len(coefficient_data)} configurations")

# Show available versions
if coefficient_data:
    unique_versions = set(v['version_num'] for v in coefficient_data.values())
    unique_configs = set(v['config'] for v in coefficient_data.values())
    print(f"✓ Unique versions found: {sorted(unique_versions)}")
    print(f"✓ Unique configurations: {sorted(unique_configs)}")

# Create summary DataFrame with ALL data for Excel - sorted by turbulence model, then AoA
coeff_summary = []
for (config, aoa), coeff in sorted(coefficient_data.items(), 
                                    key=lambda x: (TURBULENCE_ORDER.get(x[1]['turbulence_model'], 99),
                                                  x[0][0],
                                                  x[1]['aoa_degrees'])):
    coeff_summary.append({
        'Configuration': config,
        'Turbulence Model': coeff['turbulence_model'],
        'AoA': aoa,
        'AoA (degrees)': coeff['aoa_degrees'],
        'Geometry': coeff['geometry'],
        'Grid': coeff['grid'],
        'C_L': f"{coeff['C_L']:.6f}",
        'C_L_std': f"{coeff['C_L_std']:.6f}",
        'C_L_COV (%)': f"{coeff['C_L_cov']:.1f}",
        'C_D': f"{coeff['C_D']:.6f}",
        'C_D_std': f"{coeff['C_D_std']:.6f}",
        'C_D_COV (%)': f"{coeff['C_D_cov']:.1f}"
    })

coeff_df = pd.DataFrame(coeff_summary)

# Save ALL coefficient data to Excel
with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    coeff_df.to_excel(writer, sheet_name='Coefficients', index=False)

print(f"✓ ALL {len(coeff_summary)} coefficient configurations saved to Excel")

# Apply formatting
import time
time.sleep(0.5)
wb = load_workbook(excel_file)
ws = wb['Coefficients']

for column in ws.columns:
    max_length = 0
    column_letter = get_column_letter(column[0].column)
    for cell in column:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 25)
    ws.column_dimensions[column_letter].width = adjusted_width

for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment
    cell.border = border_style

for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
    fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
    for cell in row:
        cell.alignment = data_alignment
        cell.border = border_style
        cell.fill = fill_color

ws.row_dimensions[1].height = 30
wb.save(excel_file)
wb.close()


print(f"✓ Coefficient data saved to Excel sheet: 'Coefficients'")


✓ Coefficients calculated for 16 configurations
✓ Unique versions found: ['NG']
✓ Unique configurations: ['3.1.1.NG.10', '3.1.1.NG.11', '3.1.1.NG.12', '3.1.1.NG.13', '3.1.1.NG.14', '3.1.1.NG.15', '3.1.1.NG.16', '3.1.1.NG.17', '3.1.1.NG.18', '3.1.1.NG.19', '3.1.1.NG.20', '3.1.1.NG.5', '3.1.1.NG.6', '3.1.1.NG.7', '3.1.1.NG.8', '3.1.1.NG.9']
✓ ALL 16 coefficient configurations saved to Excel
✓ Coefficient data saved to Excel sheet: 'Coefficients'
✓ Coefficient data saved to Excel sheet: 'Coefficients'


In [39]:
# ==================== CREATE C_L vs AoA and C_D vs AoA GRAPHS ====================
# Create folder structure: coefficient_graphs / turbulence_model / config

# Get all unique base configurations (without AoA - first 4 parts)
unique_base_configs = set()
for (config, aoa), coeff in coefficient_data.items():
    parts = config.split('.')
    if len(parts) >= 4:
        base_config = '.'.join(parts[:4])  # e.g., "4.3.1.2"
        unique_base_configs.add(base_config)

# Group configurations by turbulence model
configs_by_turbulence = defaultdict(list)
for base_config in unique_base_configs:
    parts = base_config.split('.')
    turb_num = parts[POSITION_MAP['turbulence']] if len(parts) > POSITION_MAP['turbulence'] else 'unknown'
    turb_name = VALUE_MAPPINGS['turbulence'].get(int(turb_num) if turb_num.isdigit() else turb_num, f"Turbulence_{turb_num}")
    configs_by_turbulence[turb_name].append(base_config)

print(f"\n✓ Found {len(unique_base_configs)} unique configurations organized by turbulence model:")
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"\n  {turb_name}/")
    for cfg in sorted(configs_by_turbulence[turb_name]):
        print(f"    - {cfg}/")

# Define colors and markers for different turbulence models
model_styles = {
    'SST': {'color': '#1f77b4', 'marker': 'o', 'label': 'SST'},
    'RNG': {'color': '#ff7f0e', 'marker': 's', 'label': 'RNG k-ε'},
    'RSM': {'color': '#2ca02c', 'marker': '^', 'label': 'RSM'},
    'k-epsilon': {'color': '#d62728', 'marker': 'D', 'label': 'k-ε'}
}

# Create graphs organized by turbulence model
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"\n{'='*70}")
    print(f"TURBULENCE MODEL: {turb_name}")
    print(f"{'='*70}")
    
    for base_config in sorted(configs_by_turbulence[turb_name]):
        # Create directory: coefficient_graphs / turbulence_model / config
        config_graphs_dir = os.path.join(output_dir, "coefficient_graphs", turb_name, base_config)
        os.makedirs(config_graphs_dir, exist_ok=True)
        
        # Get descriptive info for this config
        parts = base_config.split('.')
        geometry_name = VALUE_MAPPINGS['geometry'].get(int(parts[0]) if parts[0].isdigit() else parts[0], f"Geometry_{parts[0]}")
        mesh_name = VALUE_MAPPINGS['mesh'].get(int(parts[1]) if parts[1].isdigit() else parts[1], f"Mesh_{parts[1]}")
        version_name = VALUE_MAPPINGS['version'].get(int(parts[3]) if parts[3].isdigit() else parts[3], f"Version_{parts[3]}")
        
        config_description = f"{geometry_name}, {mesh_name}, {turb_name}, {version_name}"
        
        print(f"\n  Creating graphs for {base_config}")
        print(f"    ({config_description})")
        
        # Filter coefficient data for this base configuration
        config_coeff_data = {}
        for (config, aoa), coeff in coefficient_data.items():
            config_parts = config.split('.')
            if len(config_parts) >= 4:
                this_base = '.'.join(config_parts[:4])
                if this_base == base_config:
                    config_coeff_data[(config, aoa)] = coeff
        
        if not config_coeff_data:
            print(f"    ⚠ No data found for {base_config}")
            continue
        
        # Organize data by AoA for plotting
        aoa_list = []
        C_L_list = []
        C_D_list = []
        C_L_std_list = []
        C_D_std_list = []
        
        for (config, aoa), coeff in config_coeff_data.items():
            aoa_list.append(coeff['aoa_degrees'])
            C_L_list.append(coeff['C_L'])
            C_D_list.append(coeff['C_D'])
            C_L_std_list.append(coeff['C_L_std'])
            C_D_std_list.append(coeff['C_D_std'])
        
        # Sort by AoA
        combined = list(zip(aoa_list, C_L_list, C_D_list, C_L_std_list, C_D_std_list))
        combined.sort(key=lambda x: x[0])
        
        aoa_vals = np.array([x[0] for x in combined])
        C_L_vals = np.array([x[1] for x in combined])
        C_D_vals = np.array([x[2] for x in combined])
        C_L_std_vals = np.array([x[3] for x in combined])
        C_D_std_vals = np.array([x[4] for x in combined])
        
        # Get style based on turbulence model
        style = model_styles.get(turb_name, {'color': '#1f77b4', 'marker': 'o', 'label': turb_name})
        
        # ==================== PLOT 1: C_L vs AoA ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(aoa_vals, C_L_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax.set_xlabel('Angle of Attack (degrees)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Lift Coefficient ($C_L$)', fontsize=14, fontweight='bold')
        ax.set_title(f'Lift Coefficient vs Angle of Attack\n{base_config}: {config_description}', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=12, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=11)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_L_vs_AoA.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 2: C_D vs AoA ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(aoa_vals, C_D_vals, yerr=C_D_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax.set_xlabel('Angle of Attack (degrees)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Drag Coefficient ($C_D$)', fontsize=14, fontweight='bold')
        ax.set_title(f'Drag Coefficient vs Angle of Attack\n{base_config}: {config_description}', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=12, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=11)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_D_vs_AoA.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 3: C_L vs C_D (Drag Polar) ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(C_D_vals, C_L_vals, xerr=C_D_std_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        # Add AoA annotations at each point
        for i, aoa in enumerate(aoa_vals):
            ax.annotate(f"{int(aoa)}°", 
                       (C_D_vals[i], C_L_vals[i]),
                       textcoords="offset points", xytext=(8, 5),
                       fontsize=10, fontweight='bold', color=style['color'])
        
        ax.set_xlabel('Drag Coefficient ($C_D$)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Lift Coefficient ($C_L$)', fontsize=14, fontweight='bold')
        ax.set_title(f'Drag Polar ($C_L$ vs $C_D$)\n{base_config}: {config_description}', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=12, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=11)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "Drag_Polar.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 4: Combined C_L and C_D vs AoA ====================
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
        
        # Left plot: C_L vs AoA
        ax1.errorbar(aoa_vals, C_L_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax1.set_xlabel('Angle of Attack (degrees)', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Lift Coefficient ($C_L$)', fontsize=14, fontweight='bold')
        ax1.set_title(f'Lift Coefficient vs AoA\n{base_config}', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--')
        ax1.legend(fontsize=11, loc='best', framealpha=0.9)
        ax1.tick_params(labelsize=11)
        
        # Right plot: C_D vs AoA
        ax2.errorbar(aoa_vals, C_D_vals, yerr=C_D_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax2.set_xlabel('Angle of Attack (degrees)', fontsize=14, fontweight='bold')
        ax2.set_ylabel('Drag Coefficient ($C_D$)', fontsize=14, fontweight='bold')
        ax2.set_title(f'Drag Coefficient vs AoA\n{base_config}', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3, linestyle='--')
        ax2.legend(fontsize=11, loc='best', framealpha=0.9)
        ax2.tick_params(labelsize=11)
        
        plt.suptitle(config_description, fontsize=12, y=0.02)
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_L_C_D_Combined.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"    ✓ All 4 graphs saved to: {turb_name}/{base_config}/")

print("\n" + "=" * 80)
print("COEFFICIENT GRAPHS COMPLETE")
print(f"✓ Folder structure: coefficient_graphs / turbulence_model / config")
print(f"✓ Location: {os.path.join(output_dir, 'coefficient_graphs')}")
print(f"\nFolder structure:")
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"  {turb_name}/")
    for cfg in sorted(configs_by_turbulence[turb_name]):
        print(f"    └── {cfg}/")
print("\n✓ Each config folder contains: C_L_vs_AoA.png, C_D_vs_AoA.png, Drag_Polar.png, C_L_C_D_Combined.png")
print("=" * 80)


✓ Found 1 unique configurations organized by turbulence model:

  Turbulence_1/
    - 3.1.1.NG/

TURBULENCE MODEL: Turbulence_1

  Creating graphs for 3.1.1.NG
    (Geometry_3, Mesh_1, Turbulence_1, Version_NG)
    ✓ All 4 graphs saved to: Turbulence_1/3.1.1.NG/

COEFFICIENT GRAPHS COMPLETE
✓ Folder structure: coefficient_graphs / turbulence_model / config
✓ Location: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\coefficient_graphs

Folder structure:
  Turbulence_1/
    └── 3.1.1.NG/

✓ Each config folder contains: C_L_vs_AoA.png, C_D_vs_AoA.png, Drag_Polar.png, C_L_C_D_Combined.png
    ✓ All 4 graphs saved to: Turbulence_1/3.1.1.NG/

COEFFICIENT GRAPHS COMPLETE
✓ Folder structure: coefficient_graphs / turbulence_model / config
✓ Location: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\coefficient_graphs

Folder structure:
  Turbulence_1/
    └── 3.1.1.NG/

✓ Each confi